# H&M Tabular-Only Full Training Baseline

Bu notebook, Perseptron projesindeki multimodal deneyin **tabular-only baseline** modelini eğitmek için hazırlanmıştır. Bu model görsel embedding veya müşteri görsel profili kullanmaz; yalnızca müşteri bilgileri ve ürün metadata'sı üzerinden müşteri-ürün satın alma olasılığını tahmin eder.

Amaç, late fusion modelinin gerçekten tabular bilgiye ek olarak görsel bilgiden faydalanıp faydalanmadığını karşılaştırmaktır. Bu nedenle eğitim mantı late fusion notebook ile aynı tutulur: full positive pair havuzu kullanılır, her pozitif örnek için negatif örnekler üretilir ve veri RAM'e tek parça alınmak yerine chunk'lar halinde işlenir.


## 1. Kurulum ve Deney Ayarları

Bu hücrede Kaggle input yolları, model hiperparametreleri ve tabular özellik listeleri tanımlanır. Eğer daha önce yüklenen embedding ID dataset'i mevcutsa, tabular baseline da late fusion ile aynı ürün evreninde eğitilir. Bu adım görsel embedding'i feature olarak kullanmaz; sadece adil karşılaştırma için aynı article kapsamını korur.


In [1]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

FAST_RUN = False
RANDOM_SEED = 42
WORK_DIR = Path("/kaggle/working")

STREAM_POSITIVE_CHUNK_SIZE = 200_000
STREAM_EPOCHS = 3
STREAM_BATCH_SIZE = 4096
VALIDATION_POSITIVE_ROWS = 120_000
NEGATIVES_PER_POSITIVE = 2
LEARNING_RATE = 1e-3

MODEL_PATH = WORK_DIR / "tabular_only_streaming_full.pt"
RESULTS_PATH = WORK_DIR / "tabular_only_full_results.csv"

# Optional: use the same article universe as the full image embedding run.
UPLOADED_EMBEDDING_IDS_DIR = Path("/kaggle/input/datasets/smoke78/article-image-embedding-ids-popular-csv")

NUMERIC_FEATURES = [
    "FN",
    "Active",
    "age",
    "product_code",
    "product_type_no",
    "graphical_appearance_no",
    "colour_group_code",
    "perceived_colour_value_id",
    "perceived_colour_master_id",
    "department_no",
    "index_group_no",
    "section_no",
    "garment_group_no",
]

CATEGORICAL_FEATURES = [
    "club_member_status",
    "fashion_news_frequency",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "perceived_colour_value_name",
    "perceived_colour_master_name",
    "department_name",
    "index_code",
    "index_name",
    "index_group_name",
    "section_name",
    "garment_group_name",
]


def log(message: str) -> None:
    print(message, flush=True)


def get_device(stage: str) -> torch.device:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        name = torch.cuda.get_device_name(0)
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        log(f"[{stage}] CUDA available: True | GPU: {name} | allocated={allocated:.2f}GB | reserved={reserved:.2f}GB")
    else:
        log(f"[{stage}] CUDA available: False | running on CPU")
    return device


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_hm_input_dir() -> Path:
    candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/input/competitions")]:
        if not root.exists():
            continue
        for path in root.iterdir():
            if path.is_dir() and (path / "transactions_train.csv").exists():
                candidates.append(path)
            if path.is_dir():
                for child in path.iterdir():
                    if child.is_dir() and (child / "transactions_train.csv").exists():
                        candidates.append(child)
    if not candidates:
        available = "\n".join(str(path) for path in Path("/kaggle/input").iterdir())
        raise FileNotFoundError(f"Could not find H&M input directory. Available entries:\n{available}")
    return candidates[0]


INPUT_DIR = find_hm_input_dir()
log(f"Using input directory: {INPUT_DIR}")

TRANSACTIONS_PATH = INPUT_DIR / "transactions_train.csv"
CUSTOMERS_PATH = INPUT_DIR / "customers.csv"
ARTICLES_PATH = INPUT_DIR / "articles.csv"


Using input directory: /kaggle/input/competitions/h-and-m-personalized-fashion-recommendations


## 2. Yardımcı Fonksiyonlar ve Tabular Model

Bu bölümde veri yükleme, opsiyonel article evreni kısıtlama, metadata encoding, PyTorch dataset ve tabular MLP modeli tanımlanır. Modelin görsel branch'i yoktur; sayısal ve kategorik tabular özellikler tek MLP hattında işlenir.


In [2]:
def reduce_transactions_memory(transactions: pd.DataFrame) -> pd.DataFrame:
    transactions["article_id"] = transactions["article_id"].astype(str).str.zfill(10)
    transactions["price"] = transactions["price"].astype("float32")
    transactions["sales_channel_id"] = transactions["sales_channel_id"].astype("int8")
    return transactions


def load_raw_data():
    log("Loading raw H&M CSV files...")
    transactions = pd.read_csv(TRANSACTIONS_PATH, dtype={"article_id": str})
    customers = pd.read_csv(CUSTOMERS_PATH)
    articles = pd.read_csv(ARTICLES_PATH, dtype={"article_id": str})

    transactions = reduce_transactions_memory(transactions)
    articles["article_id"] = articles["article_id"].astype(str).str.zfill(10)

    log(f"Transactions: {len(transactions):,}")
    log(f"Customers: {len(customers):,}")
    log(f"Articles: {len(articles):,}")
    return transactions, customers, articles


def first_file_in_dir(path: Path) -> Path | None:
    if path.is_file():
        return path
    if not path.exists():
        return None
    files = [candidate for candidate in path.iterdir() if candidate.is_file()]
    return files[0] if files else None


def load_optional_article_universe() -> set[str] | None:
    ids_file = first_file_in_dir(UPLOADED_EMBEDDING_IDS_DIR)
    if ids_file is None:
        log("No uploaded embedding-id dataset found. Tabular baseline will use all H&M articles.")
        return None

    log(f"Loading article universe from uploaded ids: {ids_file}")
    ids = pd.read_csv(ids_file, dtype={"article_id": str})["article_id"].astype(str).str.zfill(10)
    article_universe = set(ids.tolist())
    log(f"Article universe from embedding ids: {len(article_universe):,}")
    return article_universe


def build_global_metadata(customers: pd.DataFrame, articles: pd.DataFrame):
    category_maps = {}
    category_sizes = []

    for col in CATEGORICAL_FEATURES:
        if col in customers.columns:
            values = customers[col].fillna("__MISSING__").astype(str)
        else:
            values = articles[col].fillna("__MISSING__").astype(str)
        categories = sorted(values.unique().tolist())
        mapping = {value: idx + 1 for idx, value in enumerate(categories)}
        category_maps[col] = mapping
        category_sizes.append(len(mapping) + 1)

    return {
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "category_maps": category_maps,
        "category_sizes": category_sizes,
    }


def encode_tabular_chunk(data: pd.DataFrame, metadata: dict, numeric_mean=None, numeric_std=None):
    numeric = data[NUMERIC_FEATURES].copy()
    for col in NUMERIC_FEATURES:
        numeric[col] = pd.to_numeric(numeric[col], errors="coerce")
        numeric[col] = numeric[col].fillna(numeric[col].median())

    numeric_values = numeric.astype("float32").to_numpy()
    if numeric_mean is None:
        numeric_mean = numeric_values.mean(axis=0)
        numeric_std = numeric_values.std(axis=0)
        numeric_std[numeric_std == 0] = 1.0
    numeric_values = (numeric_values - numeric_mean) / numeric_std

    categorical_arrays = []
    for col in CATEGORICAL_FEATURES:
        mapping = metadata["category_maps"][col]
        values = data[col].fillna("__MISSING__").astype(str)
        categorical_arrays.append(values.map(mapping).fillna(0).astype("int64").to_numpy())

    return numeric_values, np.stack(categorical_arrays, axis=1), numeric_mean, numeric_std


class TabularStreamDataset(Dataset):
    def __init__(self, numeric, categorical, labels):
        self.numeric = torch.tensor(numeric, dtype=torch.float32)
        self.categorical = torch.tensor(categorical, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.numeric[idx], self.categorical[idx], self.labels[idx]


class TabularOnlyMLP(nn.Module):
    def __init__(self, numeric_dim: int, category_sizes: list[int]):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(size, min(50, max(4, (size + 1) // 2)))
            for size in category_sizes
        ])
        categorical_dim = sum(layer.embedding_dim for layer in self.embeddings)
        self.network = nn.Sequential(
            nn.Linear(numeric_dim + categorical_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

    def forward(self, numeric, categorical):
        embedded = [layer(categorical[:, i]) for i, layer in enumerate(self.embeddings)]
        x = torch.cat([numeric, *embedded], dim=1)
        return self.network(x).squeeze(1)


def prepare_pair_chunk(positive_chunk, chunk_id, article_pool, rng, customers, articles):
    positives = positive_chunk.copy()
    positives["label"] = 1

    neg_customers = np.repeat(positives["customer_id"].values, NEGATIVES_PER_POSITIVE)
    neg_articles = rng.choice(article_pool, size=len(neg_customers), replace=True)
    negatives = pd.DataFrame({"customer_id": neg_customers, "article_id": neg_articles, "label": 0})

    data = pd.concat([positives, negatives], ignore_index=True)
    data = data.drop_duplicates(["customer_id", "article_id", "label"])
    data = data.sample(frac=1.0, random_state=RANDOM_SEED + chunk_id).reset_index(drop=True)

    data = data.merge(customers, on="customer_id", how="left")
    data = data.merge(articles, on="article_id", how="left")

    del positives, negatives, neg_customers, neg_articles
    gc.collect()
    return data


def train_one_chunk(model, optimizer, criterion, data, metadata, numeric_mean, numeric_std, device):
    numeric, categorical, _, _ = encode_tabular_chunk(data, metadata, numeric_mean, numeric_std)
    dataset = TabularStreamDataset(numeric, categorical, data["label"].to_numpy(dtype="float32"))
    loader = DataLoader(dataset, batch_size=STREAM_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"))

    model.train()
    losses = []
    for numeric_b, categorical_b, labels_b in loader:
        numeric_b = numeric_b.to(device, non_blocking=True)
        categorical_b = categorical_b.to(device, non_blocking=True)
        labels_b = labels_b.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(numeric_b, categorical_b)
        loss = criterion(logits, labels_b)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    del dataset, loader, numeric, categorical
    gc.collect()
    return float(np.mean(losses))


def evaluate_model(model, data, metadata, numeric_mean, numeric_std, device):
    numeric, categorical, _, _ = encode_tabular_chunk(data, metadata, numeric_mean, numeric_std)
    dataset = TabularStreamDataset(numeric, categorical, data["label"].to_numpy(dtype="float32"))
    loader = DataLoader(dataset, batch_size=STREAM_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(device.type == "cuda"))

    model.eval()
    probs = []
    labels_out = []
    with torch.no_grad():
        for numeric_b, categorical_b, labels_b in tqdm(loader, desc="Validation", leave=False):
            numeric_b = numeric_b.to(device, non_blocking=True)
            categorical_b = categorical_b.to(device, non_blocking=True)
            logits = model(numeric_b, categorical_b)
            probs.append(torch.sigmoid(logits).cpu().numpy())
            labels_out.append(labels_b.numpy())

    y_prob = np.concatenate(probs)
    y_true = np.concatenate(labels_out)
    y_pred = (y_prob >= 0.5).astype("int64")

    del dataset, loader, numeric, categorical
    gc.collect()
    return {
        "auc_roc": roc_auc_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
    }


## 3. Veriyi Yükleme ve Full Positive Pair Havuzunu Hazırlama

Bu hücre ham H&M tablolarını yükler ve benzersiz müşteri-ürün pozitif çiftlerini çıkarır. Eğer embedding ID dosyası mevcutsa, tabular baseline aynı article evreniyle sınırlandırılır. Bu, late fusion sonucu ile daha adil karşılaştırma yapmayı sağlar.


In [3]:
set_seed(RANDOM_SEED)
device = get_device("tabular-only startup")

transactions, customers, articles = load_raw_data()
article_universe = load_optional_article_universe()

if article_universe is not None:
    log("Restricting tabular baseline to the same article universe as the embedding run...")
    transactions = transactions[transactions["article_id"].isin(article_universe)].copy()
    articles = articles[articles["article_id"].isin(article_universe)].copy()

log(f"Transactions after article filter: {len(transactions):,}")
log(f"Articles after article filter: {len(articles):,}")

positive_pairs = transactions[["customer_id", "article_id"]].drop_duplicates().reset_index(drop=True)
log(f"Full positive pairs: {len(positive_pairs):,}")

article_pool = articles["article_id"].drop_duplicates().to_numpy()
log(f"Negative article pool: {len(article_pool):,}")


[tabular-only startup] CUDA available: False | running on CPU
Loading raw H&M CSV files...
Transactions: 31,788,324
Customers: 1,371,980
Articles: 105,542
Loading article universe from uploaded ids: /kaggle/input/datasets/smoke78/article-image-embedding-ids-popular-csv/article_image_embedding_ids_popular.csv
Article universe from embedding ids: 105,100
Restricting tabular baseline to the same article universe as the embedding run...
Transactions after article filter: 31,651,678
Articles after article filter: 105,100
Full positive pairs: 27,194,909
Negative article pool: 105,100


## 4. Streaming Tabular-Only Eğitim

Bu hücre validation setini hazırlar, tabular metadata encoding bilgisini kurar ve modeli chunk'lar üzerinden eğitir. Her epoch sonunda validation AUC ve accuracy hesaplanır. En iyi checkpoint `/kaggle/working/tabular_only_streaming_full.pt` olarak kaydedilir.


In [4]:
rng = np.random.default_rng(RANDOM_SEED)
metadata = build_global_metadata(customers, articles)

validation_positive_count = min(VALIDATION_POSITIVE_ROWS, max(1, len(positive_pairs) // 10))
val_indices = positive_pairs.sample(validation_positive_count, random_state=RANDOM_SEED).index
val_positive = positive_pairs.loc[val_indices].reset_index(drop=True)
train_positive_pairs = positive_pairs.drop(val_indices).reset_index(drop=True)

log(f"Training positive pairs: {len(train_positive_pairs):,}")
log(f"Validation positive pairs: {len(val_positive):,}")

validation_data = prepare_pair_chunk(
    val_positive,
    chunk_id=999_999,
    article_pool=article_pool,
    rng=rng,
    customers=customers,
    articles=articles,
)

log("Fitting numeric normalization from validation/calibration data...")
_, _, numeric_mean, numeric_std = encode_tabular_chunk(validation_data, metadata)
metadata["numeric_mean"] = numeric_mean
metadata["numeric_std"] = numeric_std

model = TabularOnlyMLP(
    numeric_dim=len(NUMERIC_FEATURES),
    category_sizes=metadata["category_sizes"],
).to(device)

pos_weight = torch.tensor([NEGATIVES_PER_POSITIVE], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

best_auc = -1.0
best_state = None
num_chunks = int(np.ceil(len(train_positive_pairs) / STREAM_POSITIVE_CHUNK_SIZE))
log(f"Streaming chunks per epoch: {num_chunks}")

for epoch in range(1, STREAM_EPOCHS + 1):
    log(f"===== Epoch {epoch}/{STREAM_EPOCHS} =====")
    epoch_losses = []
    shuffled = train_positive_pairs.sample(frac=1.0, random_state=RANDOM_SEED + epoch).reset_index(drop=True)

    for chunk_id, start in enumerate(tqdm(range(0, len(shuffled), STREAM_POSITIVE_CHUNK_SIZE), desc=f"Epoch {epoch} chunks")):
        end = min(start + STREAM_POSITIVE_CHUNK_SIZE, len(shuffled))
        positive_chunk = shuffled.iloc[start:end].copy()

        chunk_data = prepare_pair_chunk(
            positive_chunk,
            chunk_id=chunk_id + epoch * 100_000,
            article_pool=article_pool,
            rng=rng,
            customers=customers,
            articles=articles,
        )

        loss = train_one_chunk(model, optimizer, criterion, chunk_data, metadata, numeric_mean, numeric_std, device)
        epoch_losses.append(loss)
        log(f"Epoch {epoch} | chunk {chunk_id + 1}/{num_chunks} | rows={len(chunk_data):,} | loss={loss:.4f}")

        del positive_chunk, chunk_data
        gc.collect()

    metrics = evaluate_model(model, validation_data, metadata, numeric_mean, numeric_std, device)
    log(
        f"Epoch {epoch}/{STREAM_EPOCHS} | "
        f"loss={np.mean(epoch_losses):.4f} | "
        f"val_auc={metrics['auc_roc']:.4f} | "
        f"val_acc={metrics['accuracy']:.4f}"
    )

    if metrics["auc_roc"] > best_auc:
        best_auc = metrics["auc_roc"]
        best_state = {
            "model_state_dict": model.state_dict(),
            "metadata": metadata,
            "validation_metrics": metrics,
            "config": {
                "stream_positive_chunk_size": STREAM_POSITIVE_CHUNK_SIZE,
                "stream_epochs": STREAM_EPOCHS,
                "stream_batch_size": STREAM_BATCH_SIZE,
                "negatives_per_positive": NEGATIVES_PER_POSITIVE,
                "learning_rate": LEARNING_RATE,
                "full_positive_pairs": len(positive_pairs),
                "validation_positive_rows": len(val_positive),
                "article_universe_size": len(article_pool),
            },
        }
        torch.save(best_state, MODEL_PATH)
        log(f"Saved best checkpoint: {MODEL_PATH}")

pd.DataFrame([
    {
        "model": "tabular_only_streaming_full",
        "auc_roc": best_state["validation_metrics"]["auc_roc"],
        "accuracy": best_state["validation_metrics"]["accuracy"],
        "full_positive_pairs": len(positive_pairs),
        "validation_positive_rows": len(val_positive),
        "article_universe_size": len(article_pool),
        "checkpoint": str(MODEL_PATH),
    }
]).to_csv(RESULTS_PATH, index=False)

log(f"Tabular-only training complete. Best validation AUC: {best_auc:.4f}")
log(f"Saved results: {RESULTS_PATH}")


Training positive pairs: 27,074,909
Validation positive pairs: 120,000
Fitting numeric normalization from validation/calibration data...
Streaming chunks per epoch: 136
===== Epoch 1/3 =====


Epoch 1 chunks:   0%|          | 0/136 [00:00<?, ?it/s]

Epoch 1 | chunk 1/136 | rows=599,994 | loss=0.7396


Epoch 1 chunks:   1%|          | 1/136 [00:20<46:23, 20.62s/it]

Epoch 1 | chunk 2/136 | rows=599,997 | loss=0.7165


...

Epoch 3 | chunk 135/136 | rows=599,991 | loss=0.6393


Epoch 3 chunks:  99%|█████████▉| 135/136 [45:51<00:20, 20.16s/it]

Epoch 3 | chunk 136/136 | rows=224,726 | loss=0.6372


Epoch 3 chunks: 100%|██████████| 136/136 [45:59<00:00, 20.29s/it]


Epoch 3/3 | loss=0.6399 | val_auc=0.8550 | val_acc=0.7636
Saved best checkpoint: /kaggle/working/tabular_only_streaming_full.pt
Tabular-only training complete. Best validation AUC: 0.8550
Saved results: /kaggle/working/tabular_only_full_results.csv
